# 📂 Procesamiento del Dataset de Imágenes
---

Este cuaderno documenta **todo el proceso de construcción del dataset de imágenes** utilizado para entrenar los modelos de detección de deepfakes: desde la elección del dataset original hasta la estructura final con las imágenes listas para el entrenamiento.

No se trata de código ejecutable en este punto, sino de una **explicación detallada y razonada** de cada decisión tomada durante el preprocesamiento.

---

## 1. El dataset: FaceForensics++

## 1.1 ¿Por qué FaceForensics++?

FaceForensics++ es el benchmark de referencia para la detección de deepfakes. Fue publicado por Rössler et al. (2019) y contiene **1.000 vídeos reales** obtenidos de YouTube, junto con sus versiones manipuladas mediante **6 técnicas distintas de falsificación facial**:

| Técnica | Descripción |
|---|---|
| **Deepfakes** | Intercambio de caras mediante autoencoders (la técnica original de deepfake) |
| **Face2Face** | Transferencia de expresiones faciales entre dos personas |
| **FaceSwap** | Intercambio de la geometría completa de la cara |
| **NeuralTextures** | Re-síntesis de la cara mediante texturas aprendidas con redes neuronales |
| **FaceShifter** | Intercambio de identidad de alta fidelidad con preservación de oclusiones |
| **DeepFakeDetection** | Dataset adicional creado por Google para el reto de detección de deepfakes |

Su uso está ampliamente extendido en la literatura científica, lo que permite comparar los resultados del TFG con el estado del arte de forma directa.

---

### 1.2 El problema con el dataset completo: 4 TB inviables

La versión original de FaceForensics++ en calidad RAW ocupa aproximadamente **4 TB** de almacenamiento. Descargar y procesar ese volumen de datos es completamente inviable en el contexto de un TFG por varias razones:

- ❌ Google Drive tiene un límite de **100 GB** de almacenamiento gratuito para estudiantes.
- ❌ Google Colab no permite sesiones lo suficientemente largas como para procesar ese volumen.
- ❌ El ancho de banda necesario haría que la descarga tardara días.
- ❌ Los vídeos RAW no comprimidos son innecesariamente grandes para el objetivo del modelo.
- ❌ Mi ordenador no tiene tanto almacenamiento

Se buscó una alternativa que mantuviera la calidad visual suficiente para que el modelo pudiera aprender, pero con un tamaño de fichero manejable.


---
## 2. La versión C23: alta calidad, tamaño manejable

### 2.1 ¿Qué significa C23?

FaceForensics++ distribuye sus vídeos en **tres niveles de compresión H.264**, identificados por el parámetro CRF (*Constant Rate Factor*):

| Versión | CRF | Calidad | Tamaño aproximado | Uso recomendado |
|---|---|---|---|---|
| **RAW** | — | Original sin comprimir | ~4 TB | Investigación con recursos ilimitados |
| **C23** | 23 | Alta calidad, artefactos mínimos | ~20–40 GB | ✅ Investigación estándar |
| **C40** | 40 | Calidad reducida, muchos artefactos | ~5–10 GB | Estudios de robustez |

En H.264, el CRF controla la cantidad de información descartada durante la compresión: **a menor CRF, mayor calidad y mayor tamaño de archivo**. Un CRF de 23 es el valor que los propios autores de FaceForensics++ consideran de "alta calidad" y es la versión más usada en los papers de referencia.

### 2.2 ¿Por qué C23 y no C40?

Usar C40 habría reducido aún más el tamaño, pero introduciría **artefactos de compresión muy visibles** que podrían enmascarar las señales de manipulación o crear señales artificiales no relacionadas con el deepfake. El modelo aprendería a detectar compresión, no manipulación.

Con C23, los vídeos conservan suficiente calidad visual para que las señales de manipulación facial sean detectables, lo que hace el escenario **más realista y más exigente** para el modelo.

### 2.3 Contenido descargado

El dataset FaceForensics++ C23 descargado contenía la siguiente estructura:

```
FaceForensics++_C23/
├── original/          ← 1.000 vídeos reales (personas reales de YouTube)
├── Deepfakes/         ← 1.000 vídeos falsos con técnica Deepfakes
├── Face2Face/         ← 1.000 vídeos falsos con técnica Face2Face
├── FaceSwap/          ← 1.000 vídeos falsos con técnica FaceSwap
├── NeuralTextures/    ← 1.000 vídeos falsos con técnica NeuralTextures
├── FaceShifter/       ← 1.000 vídeos falsos con técnica FaceShifter
└── DeepFakeDetection/ ← 1.000 vídeos falsos del reto de Google
```

**Total: 7.000 vídeos** (1.000 reales + 6.000 falsos, uno por cada técnica de manipulación).

Cada vídeo tiene un nombre con formato `000.mp4`, `001.mp4`, ..., `999.mp4`, y existe una correspondencia directa entre el vídeo real `N` y sus versiones manipuladas `N` en cada carpeta de técnica.


---
## 3. Primer intento: extracción de frames directos (sin detección facial)

### 3.1 La idea inicial

La primera aproximación consistió en extraer frames directamente de los vídeos sin ningún tipo de detección o recorte facial. El proceso era simple:

1. Abrir cada vídeo con OpenCV.
2. Extraer un conjunto de frames a intervalos regulares.
3. Guardar cada frame completo como imagen `.jpg`.

Este enfoque es rápido y no requiere modelos adicionales, pero presenta un problema fundamental.

### 3.2 El problema: demasiado "ruido" visual

Al guardar el frame completo del vídeo, la imagen incluye **mucho más que la cara**:

- El fondo de la escena (paredes, muebles, exteriores...).
- El cuerpo de la persona (hombros, torso...).
- Elementos irrelevantes como texto en pantalla, otros objetos, etc.

Los modelos necesitan aprender a distinguir entre caras reales y manipuladas. Si le damos imágenes con mucho contexto irrelevante, parte de su capacidad de aprendizaje se "desperdicia" en aprender características del fondo en lugar de las de la cara.

Además, el tamaño de los frames completos es muy heterogéneo entre vídeos (distintas resoluciones), lo que complica el procesamiento.

### 3.3 Conclusión: necesitamos detectar y recortar la cara

Se descartó este enfoque y se decidió añadir una etapa de **detección facial** antes de guardar las imágenes, para que el dataset final contenga únicamente recortes de caras, normalizados en tamaño.


---
## 4. Solución: detección facial con MTCNN

### 4.1 ¿Qué es MTCNN?

MTCNN (*Multi-task Cascaded Convolutional Networks*) es un sistema de detección facial en tres etapas en cascada:

- **P-Net** (*Proposal Network*): genera candidatos de regiones donde puede haber una cara.
- **R-Net** (*Refine Network*): refina los candidatos y elimina los falsos positivos.
- **O-Net** (*Output Network*): produce la detección final con landmarks faciales.

Es especialmente robusto ante variaciones de **escala, iluminación y pose**, lo que lo hace adecuado para vídeos grabados en condiciones diversas.

### 4.2 Parámetros utilizados

```python
mtcnn = MTCNN(
    image_size  = 299,   # Tamaño de salida en píxeles (ancho y alto)
    margin      = 30,    # Píxeles extra alrededor del recuadro de la cara
    keep_all    = False, # Solo detectar la cara más prominente por frame
    post_process= False, # Devolver valores en rango [0, 255], no [-1, 1]
    device      = device # Usar GPU si está disponible
)
```

**¿Por qué `image_size=299`?**
XceptionNet, al igual que InceptionV3 del que hereda la arquitectura, espera imágenes de entrada de exactamente **299×299 píxeles**. Al configurar MTCNN con este tamaño, las caras detectadas quedan directamente en el formato requerido por el modelo, sin necesidad de un redimensionado posterior.

**¿Por qué `margin=30`?**
Un margen de 30 píxeles alrededor del bounding box de la cara asegura que se incluye contexto periférico (orejas, frente, mentón), que puede contener señales de manipulación en las costuras de los deepfakes.

### 4.3 Proceso por cada frame

```
Frame del vídeo (BGR)
        ↓
Convertir a RGB
        ↓
MTCNN detecta la cara
        ↓
Devuelve tensor (3 × 299 × 299)
        ↓
Convertir a imagen NumPy (299 × 299 × 3)
        ↓
Guardar como .jpg
```

Si MTCNN no detecta ninguna cara en un frame (por oclusión, mal encuadre, etc.), ese frame se descarta silenciosamente y se continúa con el siguiente.


---
## 5. Número de frames extraídos por vídeo

### 5.1 La decisión: 15 frames equidistantes

De cada vídeo se extraen **15 frames**, seleccionados de forma equidistante a lo largo de la duración total del vídeo:

```python
indices = np.linspace(0, total_frames - 1, 15, dtype=int)
```

Esto significa que si un vídeo tiene 300 frames, se extraen los frames en las posiciones:
`0, 21, 42, 64, 85, 107, 128, 150, 171, 192, 214, 235, 257, 278, 299`

### 5.2 ¿Por qué 15 y no más o menos?

| Nº de frames | Ventajas | Inconvenientes |
|---|---|---|
| 5 frames | Muy ligero, rápido | Poca diversidad temporal, puede perder partes del vídeo |
| **15 frames** ✅ | Buen equilibrio diversidad/tamaño | — |
| 30+ frames | Más diversidad | Imágenes redundantes entre frames consecutivos, dataset enorme |

Con 15 frames se captura la diversidad temporal de la manipulación (que puede variar a lo largo del vídeo) sin generar imágenes casi idénticas entre sí. Los vídeos de FaceForensics++ tienen entre 150 y 1.000 frames, por lo que 15 frames equidistantes garantizan una cobertura representativa.

### 5.3 Vídeos descartados

Los vídeos con **menos de 15 frames** en total se descartan automáticamente, ya que no es posible extraer la cantidad mínima de muestras representativas. En la práctica esto afectó a una cantidad mínima de vídeos.

### 5.4 Nombre de archivo de cada imagen

Cada imagen guardada sigue la convención:

```
{técnica}_{id_video}_f{número_frame}.jpg

Ejemplos:
  original_042_f0.jpg    ← Frame 0 del vídeo real 042
  original_042_f14.jpg   ← Frame 14 (último) del vídeo real 042
  Deepfakes_042_f7.jpg   ← Frame 7 del vídeo 042 manipulado con Deepfakes
```

El uso del frame `f14` como marcador es clave para el **sistema de reanudación**: si ese archivo existe, el vídeo ya fue procesado completamente y se puede saltar en futuras ejecuciones.


---
## 6. El problema del desbalanceo y cómo se resolvió

### 6.1 El desbalanceo inicial

El dataset FaceForensics++ presenta un **desbalanceo estructural** entre clases:

- **Reales**: 1.000 vídeos (1 carpeta × 1.000 vídeos)
- **Falsos**: 6.000 vídeos (6 carpetas × 1.000 vídeos)

Si usáramos todo el dataset tal cual, habría **6 veces más imágenes falsas que reales**. Esto es un problema grave porque:

- El modelo tendería a predecir siempre "fake" para minimizar el error.
- Las métricas de accuracy serían engañosas (podría llegar al 85% sin aprender nada).
- El modelo no aprendería a reconocer caras reales correctamente.

### 6.2 La solución: selección estratégica de 1.000 falsos

Se decidió usar exactamente **1.000 vídeos falsos** para igualar los 1.000 reales, distribuyendo esa selección de forma equitativa entre las 6 técnicas de falsificación.

#### Estrategia de distribución cíclica

Para los 1.000 vídeos falsos, se usa una **asignación cíclica por ID de vídeo**:

```
Vídeo 000 → Deepfakes
Vídeo 001 → Face2Face
Vídeo 002 → FaceSwap
Vídeo 003 → NeuralTextures
Vídeo 004 → FaceShifter
Vídeo 005 → DeepFakeDetection
Vídeo 006 → Deepfakes   (reinicia el ciclo)
Vídeo 007 → Face2Face
...
```

Esto resulta en **~166-167 vídeos por técnica**, garantizando que:
1. El dataset total está **balanceado** (1.000 reales = 1.000 falsos).
2. Los falsos representan **todas las técnicas** de forma equitativa.
3. El modelo no desarrolla sesgo hacia ninguna técnica de falsificación concreta.

#### Estrategia alternativa: rangos por técnica

En el procesamiento de fakes diversificado se probó también la asignación por **rangos consecutivos** en lugar de cíclica:

```
Técnica Deepfakes        → Vídeos 000-165
Técnica Face2Face        → Vídeos 166-331
Técnica FaceSwap         → Vídeos 332-497
Técnica NeuralTextures   → Vídeos 498-663
Técnica FaceShifter      → Vídeos 664-829
Técnica DeepFakeDetection → Vídeos 830-995
```

Esta estrategia asegura que no hay solapamiento entre técnicas y que cada técnica usa identidades completamente distintas, lo que maximiza la **diversidad de personas** en el dataset.

### 6.3 Dataset final balanceado

| Clase | Vídeos seleccionados | Frames por vídeo | Imágenes (estimadas) |
|---|---|---|---|
| **Real** | 1.000 | hasta 15 | ~14.000-15.000 |
| **Fake** | 1.000 (~166/técnica) | hasta 15 | ~14.000-15.000 |
| **Total** | 2.000 | — | **~28.000-30.000** |


---
## 7. División Train / Test

### 7.1 Criterio de división

La división entre entrenamiento y test se realiza **a nivel de vídeo**, no a nivel de imagen. Esto es fundamental para evitar *data leakage*: si diferentes frames del mismo vídeo estuvieran en train y en test, el modelo podría "reconocer" el vídeo en lugar de aprender a generalizar.

El criterio es sencillo y determinista, basado en el ID numérico del vídeo:

```
IDs 000–799  →  train  (80% de los vídeos)
IDs 800–999  →  test   (20% de los vídeos)
```

Esta división **se aplica igual para reales y falsos**, garantizando que ambas clases estén representadas en las mismas proporciones en ambos subconjuntos.

### 7.2 Estructura de carpetas resultante

```
Dataset_Final_Procesado/
│
├── train/
│   ├── real/    ← Caras reales de vídeos 000-799
│   └── fake/    ← Caras falsas de vídeos 000-799
│
└── test/
    ├── real/    ← Caras reales de vídeos 800-999
    └── fake/    ← Caras falsas de vídeos 800-999
```

Esta estructura es compatible directamente con el `ImageFolder` de PyTorch y con `ImageDataGenerator` de Keras/TensorFlow, lo que facilita la carga del dataset durante el entrenamiento.

### 7.3 Distribución final de imágenes

| Subconjunto | Clase | Vídeos | Imágenes (estimadas) |
|---|---|---|---|
| **train** | real | 800 | ~11.000-12.000 |
| **train** | fake | 800 | ~11.000-12.000 |
| **test** | real | 200 | ~2.800-3.000 |
| **test** | fake | 200 | ~2.800-3.000 |
| **TOTAL** | — | 2.000 | **~28.000-30.000** |

> El número exacto de imágenes es inferior al máximo teórico (15 × vídeos) porque MTCNN no siempre detecta una cara en todos los frames: algunos frames pueden estar borrosos, con la cara parcialmente ocluida, o mal encuadrados.

### 7.4 ¿Por qué 80/20 y no 70/30?

La división 80/20 es el estándar más habitual en deep learning cuando el dataset es de tamaño moderado (~2.000 vídeos). Con 70/30 se perdería demasiada información de entrenamiento sin una ganancia significativa en la evaluación. Con 90/10, el conjunto de test sería demasiado pequeño para obtener estimaciones fiables del rendimiento.


---
## 8. Resumen del pipeline completo

```
FaceForensics++ C23 (7.000 vídeos)
│
├── original/        1.000 vídeos reales
│       └──────────────────────────────────────────────────┐
│                                                          │
├── Deepfakes/       1.000 vídeos  ─┐                      │
├── Face2Face/       1.000 vídeos  │                       │
├── FaceSwap/        1.000 vídeos  ├─ Selección de         │
├── NeuralTextures/  1.000 vídeos  │  ~166 vídeos/técnica  │
├── FaceShifter/     1.000 vídeos  │  = 1.000 fakes total  │
└── DeepFakeDetection/ 1.000 vídeos ┘                      │
                              │                            │
                              ▼                            ▼
                    1.000 vídeos FAKE            1.000 vídeos REAL
                              │                            │
                              └──────────┬─────────────────┘
                                         │
                              División por ID de vídeo
                                         │
                   ┌─────────────────────┼─────────────────────┐
                   │                                           │
              IDs 000-799                               IDs 800-999
               (80% → TRAIN)                           (20% → TEST)
                   │                                           │
         ┌─────────┴─────────┐                     ┌──────────┴──────────┐
         │                   │                     │                     │
    MTCNN (299×299)     MTCNN (299×299)        MTCNN (299×299)      MTCNN (299×299)
    15 frames/vídeo     15 frames/vídeo        15 frames/vídeo      15 frames/vídeo
         │                   │                     │                     │
    train/real/         train/fake/            test/real/           test/fake/
   ~12.000 imgs        ~12.000 imgs            ~3.000 imgs          ~3.000 imgs
```

### Decisiones clave del pipeline

| Decisión | Alternativa descartada | Motivo |
|---|---|---|
| Usar C23 | RAW (4 TB) | Inviable en cuanto a capacidad |
| MTCNN para recorte | Frames completos | Elimina ruido visual irrelevante |
| 15 frames equidistantes | Todos los frames | Equilibrio diversidad/tamaño |
| 1.000 falsos balanceados | 6.000 falsos | Evita sesgo de clase |
| División por ID de vídeo | División aleatoria de imágenes | Evita data leakage |
| imagen_size=299 | Otros tamaños | Entrada estándar de XceptionNet |
